In [1]:
import pyrosetta
import numpy as np

from benchmark import bpti_ryfyn_benchmark
from energy_calculation import evaluate_quantum_energies, evaluate_pyrosetta_energies, compare_energies
from misc import init_generator_params
from rotamer_extraction import extract_top_n_rotamers, TrackedRotamer
from h_ising_creation import extract_hamiltonian_tensors, build_ising_hamiltonian, reduce_hamiltonian
from initialisation import initialize_rosetta
from custom_qaoa import qaoa_func_generator, run_qaoa
from h_mixer import custom_xy_mixer_layer

from constants import *
from validation import validate_conformations

initialize_rosetta(pyrosetta, extra_flags="-mute all")

Initializing PyRosetta with cleaning flags: -ignore_unrecognized_res and extra flags: -mute all
┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python313.m1 2026.03+releasequarterly.5e498f1409c68ade56c8ce5842bf79e1b02e8db4 2026-01-13T13:24:11] retrieved from: http://www.pyrosetta.org


In [2]:
# Pyrosetta Relevant Code
benchmark_pose = bpti_ryfyn_benchmark()
rotamer_lib, ig, rot_sets, scorefxn = extract_top_n_rotamers(benchmark_pose)

# Generating QUBO (Quadratic Unconstrained Binary Optimisation) Model, and then reduce it
h_linear, J_quadratic = extract_hamiltonian_tensors(rotamer_lib, ig, rot_sets)
h_flex_linear, J_flex_quadratic, global_offset = reduce_hamiltonian(h_linear, J_quadratic, rotamer_lib)
generator_params = init_generator_params(h_flex_linear)
for idx in h_linear:
    print(h_linear[idx])
    print(h_flex_linear.get(idx, "None"))
    print("\n---------------------------------------\n")

# Generate the actual observable and running functions we will use in the QAOA Algorithm
H_ising = build_ising_hamiltonian(h_flex_linear, J_flex_quadratic, global_offset)
cost_function, sample_function = qaoa_func_generator(H_ising, custom_xy_mixer_layer, generator_params)

# Run the Quantum Approximate Optimisation Algorithm and sample the final parameters
final_params = run_qaoa(cost_function)
probabilities = sample_function(final_params)

# Extract the top 100 most probably conformations and check that exactly 1 rotamer for each residue is selected
top_indices = list(np.argsort(probabilities)[-TOP_CONFORMATION_COUNTS:][::-1])
valid_conformations = validate_conformations(top_indices, probabilities, generator_params)

Fragment Sequence: RYFYN
Total Residues: 5
Creating score function
Pose scored successfully!
Creating Repacking Task - Core Rotamer Optimisation Protocol
Computing One-Body and Two-Body Energies
Iterating through molten residues - determining the top rotamer positions for each amino acid
1 1
2 2
3 3
4 4
5 5
{1: 1, 2: 2, 3: 3, 4: 4, 5: 5}
initializing generator_params
{0: 1.0419909954071045, 1: 1.2052754163742065, 2: 1.2125394344329834, 3: 1.6054599285125732}
{0: 1.0419909954071045, 1: 1.2052754163742065, 2: 1.2125394344329834, 3: 1.6054599285125732}

---------------------------------------

{0: 1.4519609212875366, 1: 1.4519609212875366, 2: 2.4010090827941895, 3: 2.4010090827941895}
{0: 1.4519609212875366, 1: 1.4519609212875366, 2: 2.4010090827941895, 3: 2.4010090827941895}

---------------------------------------

{0: 1.4840223789215088, 1: 1.6721895933151245, 2: 2.909672975540161}
{0: 1.4840223789215088, 1: 1.6721895933151245, 2: 2.909672975540161}

-----------------------------------

In [3]:
def evaluate_quantum_energies(valid_conformations, h_flex, J_flex, global_offset, params):
    wire_offsets = params["wire_offsets"]

    for conformation in valid_conformations:
        bitstring = conformation["bitstring"]
        current_energy = global_offset

        # One body energies
        for seq, energies in h_flex.items():
            base_wire = wire_offsets[seq]
            for rot, e_val in energies.items():
                if bitstring[base_wire + rot] == 1:
                    current_energy += e_val

        # Two body energies
        for (seq_i, seq_j), interactions in J_flex.items():
            for (rot_i, rot_j), e_val in interactions.items():
                k = wire_offsets[seq_i] + rot_i
                l = wire_offsets[seq_j] + rot_j

                if bitstring[k] == 1 and bitstring[l] == 1:
                    current_energy += e_val

        conformation['quantum_energy'] = current_energy
    # raise NotImplementedError("Not yet implemented")

evaluate_quantum_energies(valid_conformations, h_flex_linear, J_flex_quadratic, global_offset, params=generator_params)


In [4]:
def evaluate_pyrosetta_energies(pose, valid_conformations, scorefxn, rotamer_library, params):
    for conformation in valid_conformations:

        bitstring = conformation["bitstring"]
        bio_energy, pose = evaluate_singular_pyrosetta_energy(pose, bitstring, scorefxn, rotamer_library, params)
        conformation['biological_energy'] = bio_energy
        conformation['pose'] = pose


def evaluate_singular_pyrosetta_energy(pose, bitstrings, scorefxn, rotamer_library, params):
    test_pose = pose.clone()

    seq_positions = params["seq_positions"]
    wire_offsets = params["wire_offsets"]
    rotamer_counts = params["rotamer_counts"]

    for seq in seq_positions:
        base_wire = wire_offsets[seq]
        num_rots = rotamer_counts[seq]

        residue_bits = bitstrings[base_wire : base_wire + num_rots]
        local_rotamer_idx = residue_bits.index(1)

        res_entry = rotamer_library[seq]
        rotamer_entry = res_entry[local_rotamer_idx]

        test_pose.replace_residue(seq, rotamer_entry.residue, False)

    bio_energy = scorefxn(test_pose)
    return bio_energy, test_pose

evaluate_pyrosetta_energies(benchmark_pose, valid_conformations, scorefxn, rotamer_lib, generator_params)

In [5]:
deltas = [conf['quantum_energy'] - conf['biological_energy'] for conf in valid_conformations]
deltas

[-1.9796845052155732e-07,
 -1.9796845052155732e-07,
 -7.065948715023751e-08,
 -1.8549733571404659e-07,
 -1.4722894015051224e-08,
 -4.193180931366669e-07,
 -1.3364342166255483e-07,
 -1.3364342166255483e-07,
 -1.0754650503486118e-07,
 -1.0754650503486118e-07,
 -1.520453505321484e-07,
 -3.99390899019636e-07,
 -2.0919882004477586e-07,
 2.1248037818111243e-07,
 1.7225579540536273e-06,
 -7.065948715023751e-08,
 -1.8549733571404659e-07,
 -5.818837234272678e-08,
 -1.4722894015051224e-08,
 1.1258606935626858e-07,
 -4.193180931366669e-07,
 -2.920091297653471e-07,
 -4.0684697832915617e-07,
 -2.360725366301608e-07,
 -6.334458291235023e-09,
 -6.334458291235023e-09,
 -1.2117230596686568e-07,
 -1.2117230596686568e-07,
 4.960213573212968e-08,
 4.960213573212968e-08,
 -1.0754650503486118e-07,
 -1.0754650503486118e-07,
 -9.507538933917203e-08,
 -9.507538933917203e-08,
 -4.3510236302779504e-08,
 -4.3510236302779504e-08,
 -4.0536820922199013e-07,
 -4.0536820922199013e-07,
 -1.520453505321484e-07,
 -7.5457

In [6]:
for i, conf in enumerate(valid_conformations):
    conf["org_idx"] = i
valid_conformations

[{'bitstring': [1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
  'probability': np.float64(0.9999999998846075),
  'quantum_energy': 1.0416022762656212,
  'biological_energy': 1.0416024742340717,
  'pose': <pyrosetta.rosetta.core.pose.Pose at 0x1148787b0>,
  'org_idx': 0},
 {'bitstring': [1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
  'probability': np.float64(8.793007746045295e-12),
  'quantum_energy': 1.0416022762656212,
  'biological_energy': 1.0416024742340717,
  'pose': <pyrosetta.rosetta.core.pose.Pose at 0x114755770>,
  'org_idx': 1},
 {'bitstring': [1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0],
  'probability': np.float64(8.793007725582768e-12),
  'quantum_energy': 1.0423380360007286,
  'biological_energy': 1.0423381066602158,
  'pose': <pyrosetta.rosetta.core.pose.Pose at 0x114871c70>,
  'org_idx': 2},
 {'bitstring': [1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0],
  'probability': np.float64(8.793006291691373e-12),
  'quantum_energ

In [7]:
valid_conformations.sort(key=lambda x: x['quantum_energy'])
for i, conf in enumerate(valid_conformations):
    conf["new_idx"] = i


In [8]:
idx_diffs = [abs((100 - conf["org_idx"]) - conf["new_idx"]) for conf in valid_conformations]
np.mean(idx_diffs)

np.float64(35.52)

In [9]:
bio_energy = [conf['biological_energy'] for conf in valid_conformations]
quant_energy = [conf['quantum_energy'] for conf in valid_conformations]

bio_ordering =  [int(idx) for idx in list(np.argsort(bio_energy))]
quant_ordering = [int(idx) for idx in list(np.argsort(quant_energy))]

for bio_idx, quant_idx in zip(bio_ordering, quant_ordering):
    print(f"Bio Ranking: Quantum Ranking | {bio_idx}: {quant_idx}")

Bio Ranking: Quantum Ranking | 0: 0
Bio Ranking: Quantum Ranking | 1: 1
Bio Ranking: Quantum Ranking | 2: 2
Bio Ranking: Quantum Ranking | 3: 3
Bio Ranking: Quantum Ranking | 4: 4
Bio Ranking: Quantum Ranking | 5: 5
Bio Ranking: Quantum Ranking | 6: 6
Bio Ranking: Quantum Ranking | 7: 7
Bio Ranking: Quantum Ranking | 8: 8
Bio Ranking: Quantum Ranking | 9: 9
Bio Ranking: Quantum Ranking | 10: 10
Bio Ranking: Quantum Ranking | 11: 11
Bio Ranking: Quantum Ranking | 13: 13
Bio Ranking: Quantum Ranking | 12: 12
Bio Ranking: Quantum Ranking | 14: 14
Bio Ranking: Quantum Ranking | 15: 15
Bio Ranking: Quantum Ranking | 16: 16
Bio Ranking: Quantum Ranking | 17: 17
Bio Ranking: Quantum Ranking | 18: 18
Bio Ranking: Quantum Ranking | 19: 19
Bio Ranking: Quantum Ranking | 20: 20
Bio Ranking: Quantum Ranking | 21: 21
Bio Ranking: Quantum Ranking | 22: 22
Bio Ranking: Quantum Ranking | 23: 23
Bio Ranking: Quantum Ranking | 25: 25
Bio Ranking: Quantum Ranking | 24: 24
Bio Ranking: Quantum Ranking | 2

In [10]:
rank_mismatch = [abs(a-b) for a, b in zip(bio_ordering, quant_ordering)]
print("Mean rank mismatch", np.mean(rank_mismatch))
print("Median rank mismatch", np.median(rank_mismatch))

print("Mean rank mismatch of first 10", np.mean(rank_mismatch[:10]))
print("Median rank mismatch of first 10", np.median(rank_mismatch[:10]))

print("Mean rank mismatch of last 10", np.mean(rank_mismatch[-10:]))
print("Median rank mismatch of last 10", np.median(rank_mismatch[-10:]))

# Prior results p=8, no adjoint lightning.gpu (3min) - GPU:
# Mean rank mismatch 9.16
# Median rank mismatch 6.5
# Mean rank mismatch of first 10 4.5
# Median rank mismatch of first 10 4.0
# Mean rank mismatch of last 10 0.2
# Median rank mismatch of last 10 0.0

# Prior results p=2, adjoint lightning.qubit (3min) - Mac M1:
# Mean rank mismatch 11.8
# Median rank mismatch 8.0
# Mean rank mismatch of first 10 4.5
# Median rank mismatch of first 10 4.0
# Mean rank mismatch of last 10 0.4
# Median rank mismatch of last 10 0.0

# Prior results p=8, adjoint lightning.qubit (10min) - Mac M1:
# Mean rank mismatch 8.9
# Median rank mismatch 5.5
# Mean rank mismatch of first 10 4.5
# Median rank mismatch of first 10 4.0
# Mean rank mismatch of last 10 0.6
# Median rank mismatch of last 10 0.5

# Prior results p=16, adjoint lightning.qubit (20 min) - Mac M1:
# Mean rank mismatch 12.22
# Median rank mismatch 8.5
# Mean rank mismatch of first 10 4.5
# Median rank mismatch of first 10 4.0
# Mean rank mismatch of last 10 0.6
# Median rank mismatch of last 10 0.5

# 300 epochs - 0.007 learning rate
# Prior results p=16, adjoint lightning.qubit ( min) - Mac M1:

Mean rank mismatch 0.0
Median rank mismatch 0.0
Mean rank mismatch of first 10 0.0
Median rank mismatch of first 10 0.0
Mean rank mismatch of last 10 0.0
Median rank mismatch of last 10 0.0


In [11]:
import itertools
from rotamer_extraction import TrackedRotamer

def evaluate_two_energies(pose, valid_conformations, scorefxn, rotamer_library, params, ig, rot_sets, h_linear, J_quadratic):
    for conformation in valid_conformations:

        bitstring = conformation["bitstring"]
        bio_energy, _ = evaluate_singular_pyrosetta_energy(pose, bitstring, scorefxn, rotamer_library, params)
        conformation['bio_en'] = bio_energy

        quant_enery = evaluate_quantum_energy(bitstring, rotamer_library, params, ig, rot_sets, h_linear, J_quadratic)
        conformation['quant_en'] = quant_enery


def evaluate_bio_energy(pose, bitstring, scorefxn, rotamer_library: dict[int, list[TrackedRotamer]], params):
    test_pose = pose.clone()

    seq_positions = params["seq_positions"]
    wire_offsets = params["wire_offsets"]
    rotamer_counts = params["rotamer_counts"]

    for seq in seq_positions:
        base_wire = wire_offsets[seq]
        num_rots = rotamer_counts[seq]

        residue_bits = bitstring[base_wire : base_wire + num_rots]
        local_rotamer_idx = residue_bits.index(1)

        res_entry = rotamer_library[seq]
        rotamer_entry: TrackedRotamer = res_entry[local_rotamer_idx]

        test_pose.replace_residue(seq, rotamer_entry.residue, False)

    bio_energy = scorefxn(test_pose)
    return bio_energy, test_pose

def evaluate_quantum_energy(bitstring, rotamer_library: dict[int, list[TrackedRotamer]], params, ig, rot_sets, h_linear, J_quadratic):
    seq_positions = params["seq_positions"]
    seq_to_molten = {rot_sets.moltenres_2_resid(m): m for m in range(1, rot_sets.nmoltenres() + 1)}

    wire_offsets = params["wire_offsets"]
    rotamer_counts = params["rotamer_counts"]


    def get_picked_rotamer_idx(seq_idx):
        base_wire = wire_offsets[seq_idx]
        num_rots = rotamer_counts[seq_idx]

        residue_bits = bitstring[base_wire : base_wire + num_rots]
        local_rotamer_idx = residue_bits.index(1)

        rotamer_seq_entry = rotamer_library[seq_idx]
        rotamer_entry = rotamer_seq_entry[local_rotamer_idx]

        return rotamer_entry.original_pyrosetta_index

    one_body_energies = 0
    for seq_idx in seq_positions:
        base_wire = wire_offsets[seq_idx]
        num_rots = rotamer_counts[seq_idx]

        residue_bits = bitstring[base_wire : base_wire + num_rots]
        local_rotamer_idx = residue_bits.index(1)

        single_energy = rotamer_library[seq_idx][local_rotamer_idx].one_body_energy
        alt_single_energy = h_linear[seq_idx][local_rotamer_idx]

        assert single_energy == alt_single_energy, f"Energies do not match {seq_idx}{local_rotamer_idx}"

        one_body_energies += single_energy

    alt_one_body_energies = 0

    for seq, energies in h_linear.items():
            base_wire = wire_offsets[seq]
            # print(base_wire)
            for rot, e_val in energies.items():
                # print("\t", rot, "=>", base_wire+rot)
                if bitstring[base_wire + rot] == 1:
                    # print("\t\t", "Found item in bitstring", rot, e_val)
                    alt_one_body_energies += e_val
    print(one_body_energies, alt_one_body_energies, one_body_energies == alt_one_body_energies)

    two_body_energies = 0
    for seq_i, seq_j in itertools.combinations(seq_positions, 2):
        molten_i = seq_to_molten[seq_i]
        molten_j = seq_to_molten[seq_j]

        if not ig.get_edge_exists(molten_i, molten_j): continue
        edge = ig.find_edge(molten_i, molten_j)

        rot_index_i = get_picked_rotamer_idx(seq_i)
        rot_index_j = get_picked_rotamer_idx(seq_j)

        pair_energy = edge.get_two_body_energy(rot_index_i, rot_index_j)
        two_body_energies += pair_energy

    alt_two_body_energies = 0
    for (seq_i, seq_j), interactions in J_quadratic.items():
            for (rot_i, rot_j), e_val in interactions.items():
                k = wire_offsets[seq_i] + rot_i
                l = wire_offsets[seq_j] + rot_j

                if bitstring[k] == 1 and bitstring[l] == 1:
                    alt_two_body_energies += e_val
    # print(two_body_energies, alt_two_body_energies, two_body_energies == alt_two_body_energies)

    return one_body_energies + two_body_energies


evaluate_two_energies(benchmark_pose, valid_conformations, scorefxn, rotamer_lib, generator_params, ig, rot_sets, h_flex_linear, J_quadratic)

7.542375802993774 7.542375802993774 True
7.542375802993774 7.542375802993774 True
7.542375802993774 7.542375802993774 True
7.668914079666138 7.668914079666138 True
7.925751090049744 7.925751090049744 True
7.535111784934998 7.535111784934998 True
7.535111784934998 7.535111784934998 True
7.535111784934998 7.535111784934998 True
7.661650061607361 7.661650061607361 True
7.918487071990967 7.918487071990967 True
8.491423964500427 8.491423964500427 True
8.491423964500427 8.491423964500427 True
7.935296297073364 7.935296297073364 True
7.935296297073364 7.935296297073364 True
7.935296297073364 7.935296297073364 True
7.935296297073364 7.935296297073364 True
7.730561852455139 7.730561852455139 True
7.730561852455139 7.730561852455139 True
8.061834573745728 8.061834573745728 True
8.061834573745728 8.061834573745728 True
8.061834573745728 8.061834573745728 True
8.477568626403809 8.477568626403809 True
8.48415994644165 8.48415994644165 True
8.48415994644165 8.48415994644165 True
8.318671584129333 8.

In [12]:
[abs(round(conf['bio_en'] - conf['quant_en'],5)) for conf in valid_conformations]
valid_conformations

[{'bitstring': [0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
  'probability': np.float64(8.79286850641586e-12),
  'quantum_energy': -0.9878493323922157,
  'biological_energy': -0.9878495448725939,
  'pose': <pyrosetta.rosetta.core.pose.Pose at 0x11452cbf0>,
  'org_idx': 13,
  'new_idx': 0,
  'bio_en': -0.9878495448725939,
  'quant_en': -0.9878493323922157},
 {'bitstring': [0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
  'probability': np.float64(7.73157608999208e-23),
  'quantum_energy': -0.9878493323922157,
  'biological_energy': -0.9878495448725939,
  'pose': <pyrosetta.rosetta.core.pose.Pose at 0x114aa8fb0>,
  'org_idx': 68,
  'new_idx': 1,
  'bio_en': -0.9878495448725939,
  'quant_en': -0.9878493323922157},
 {'bitstring': [0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0],
  'probability': np.float64(7.731576071586965e-23),
  'quantum_energy': -0.9871135726571083,
  'biological_energy': -0.9871139124464499,
  'pose': <pyrosetta.rosetta.core.pose.Pose

In [16]:
for i, conf in enumerate(valid_conformations):
    print(i, conf['bitstring'])

0 [0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0]
1 [0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0]
2 [0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0]
3 [0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0]
4 [0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0]
5 [0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0]
6 [0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0]
7 [0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0]
8 [0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0]
9 [0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0]
10 [0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0]
11 [0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0]
12 [0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0]
13 [0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0]
14 [0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0]
15 [0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0]
16 [0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 

In [17]:
for i in [29, 35, 45, 47, 93, 94, 82]:
    conf = valid_conformations[i]
    print(i, conf['bitstring'])
    pyrosetta.rosetta.core.io.pdb.dump_pdb(conf['pose'], f"pose_variant_{i}.pdb")

29 [0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0]
35 [0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0]
45 [1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0]
47 [0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0]
93 [1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0]
94 [1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0]
82 [1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1]


In [18]:
for i, conf in enumerate(valid_conformations):
    pyrosetta.rosetta.core.io.pdb.dump_pdb(conf['pose'], f"poses/pose_variant_{i}.pdb")